In [0]:
# Mack's Chain-Ladder Method — Variance Estimation & Confidence Intervals
# This notebook implements Mack's (1993, 1994) distribution-free framework
# for estimating the prediction error of chain-ladder reserve estimates.
# **Key question:** *How uncertain are our ultimate loss estimates?*
# Chain-ladder produces point estimates. Mack's Method adds:
# - **MSEP** (Mean Squared Error of Prediction) per accident year
# - **Process vs Parameter variance** decomposition
# - **Confidence intervals** around ultimate loss
# **Input:** `mart_incurred_triangle` from the Medium-1 pipeline (dbt + DuckDB).
# **References:**
# - Mack, T. (1993). ASTIN Bulletin, 23(2), 213–225.
# - Mack, T. (1994). Insurance: Mathematics and Economics, 15(2–3), 133–138.

## 1. Setup

import importlib
import mack_utils
importlib.reload(mack_utils)

from mack_utils import (
    load_triangle,
    compute_ldf,
    compute_sigma2,
    extrapolate_sigma2,
    compute_msep,
    compute_ci
)

triangle = load_triangle("test_triangle.csv")
ldf = compute_ldf(triangle)
sigma2 = compute_sigma2(triangle, ldf)
sigma2 = extrapolate_sigma2(sigma2)
msep_df = compute_msep(triangle, ldf, sigma2)

# -------------------------
# Load inputs
# -------------------------
triangle = load_triangle("test_triangle.csv")

full_path_ldf = os.path.join(os.getcwd(), "data","expected_ldf.csv")
full_path_sigma2 = os.path.join(os.getcwd(), "data","expected_sigma2.csv")
full_path_msep = os.path.join(os.getcwd(), "data","expected_msep.csv")
expected_ldf = pd.read_csv(full_path_ldf, index_col="Dev")["ldf"]
expected_sigma2 = pd.read_csv(full_path_sigma2, index_col="Dev")["sigma2"]
expected_msep = pd.read_csv(full_path_msep)

# -------------------------
# Compute actual values
# -------------------------
ldf = compute_ldf(triangle)
sigma2 = extrapolate_sigma2(compute_sigma2(triangle, ldf))
msep_df = compute_msep(triangle, ldf, sigma2)

# -------------------------
# Compare
# -------------------------
print("=== LDF Comparison ===")
print("Computed:\n", ldf)
print("Expected:\n", expected_ldf)
print("Match:", ldf.round(6).equals(expected_ldf.round(6)))

print("\n=== sigma² Comparison ===")
print("Computed:\n", sigma2)
print("Expected:\n", expected_sigma2)
print("Match:", sigma2.round(6).equals(expected_sigma2.round(6)))

print("\n=== MSEP Comparison ===")
print("Computed:\n", msep_df[["AY", "ultimate", "msep"]])
print("Expected:\n", expected_msep)
msep_match = (
    msep_df["ultimate"].round(4).equals(expected_msep["ultimate"].round(4)) and
    msep_df["msep"].round(3).equals(expected_msep["msep"].round(3))
)
print("Match:", msep_match)


In [0]:

## 2. Load Incurred Triangle
# The triangle is exported from `mart_incurred_triangle` (Medium-1 dbt pipeline).
# Each row is an accident year; columns are cumulative incurred at each development year.
# Lower-right NaN cells represent future development periods not yet observed.

# Export the incurred triangle + import the file / clean the data.
df = spark.table("insurance_dbt.mart_incurred_triangle").toPandas()
path = os.path.join(os.getcwd(), "data","mart_incurred_triangle.csv")
df.to_csv(path, index=False)

import pandas as pd

path = os.path.join(os.getcwd(), "data","mart_incurred_triangle.csv")
df = pd.read_csv(path)

# incurred = paid + reserve
df["incurred"] = df["paid_to_date"] + df["reserve_amount"]

# pivot to wide triangle
triangle = df.pivot_table(
    index="accident_year",
    columns="dev_year",
    values="incurred",
    aggfunc="sum"
)

# rename columns to DY1, DY2, ...
triangle.columns = [f"DY{int(c)}" for c in triangle.columns]
triangle = triangle.reset_index().rename(columns={"accident_year": "AY"})

triangle

In [0]:
## 3. LDF Estimation (Volume-Weighted)
# Mack's framework requires **volume-weighted** LDFs, unlike the simple average
# used in Medium-1. Volume-weighted LDF gives more weight to larger accident years
# and is mathematically consistent with Mack's variance structure.
# $$\hat{f}_j = \frac{\sum_i C_{i,j+1}}{\sum_i C_{i,j}}$$
ldf = compute_ldf(triangle)
print("Volume-Weighted LDFs:")
print(ldf.to_string())

In [0]:
## 4. Variance Estimation (σ²)
# σ² measures how much individual link ratios deviate from the volume-weighted LDF.
# High σ² at early development years is expected — immature claims are volatile.
# The last development year cannot be estimated directly (only 1 observation),
# so we apply Mack's extrapolation convention:
# $$\hat{\sigma}^2_{J-1} = \min\left(\frac{\hat{\sigma}^2_{J-2} \cdot \hat{\sigma}^2_{J-2}}{\hat{\sigma}^2_{J-3}},\ \hat{\sigma}^2_{J-2}\right)$$

sigma2_raw = compute_sigma2(triangle, ldf)
sigma2 = extrapolate_sigma2(sigma2_raw)

print("σ² Estimates:")
print(sigma2.to_string())
print(f"\nσ²[{sigma2.index[-1]}] extrapolated: {pd.isna(sigma2_raw.iloc[-1])}")

In [0]:
## 5. MSEP Calculation
# MSEP decomposes ultimate loss uncertainty into two sources:
# | Component | Meaning | Driven by |
# |-----------|---------|-----------|
# | **Process variance** | Inherent randomness in future development | Portfolio risk |
# | **Parameter variance** | Uncertainty from estimating LDFs | Data scarcity |
# $$\text{MSEP}(\hat{U}_i) = \hat{U}_i^2 \sum_{j=k_i}^{J-1} \frac{\hat{\sigma}^2_j}{\hat{f}_j^2} \left(\frac{1}{\hat{C}_{i,j}} + \frac{1}{S_j}\right)$$

msep_df = compute_msep(triangle, ldf, sigma2)
msep_df

In [0]:
## 6. Confidence Intervals (95%)
# Normal approximation: $\hat{U}_i \pm z_{\alpha/2} \times \sqrt{\text{MSEP}_i}$
# Wider intervals for recent accident years reflect greater uncertainty —
# not model failure, but the reality of incomplete information.

for _, row in msep_df.iterrows():
    lo, hi = compute_ci(row['ultimate'], row['msep'], level=0.95)
    print(f"AY {int(row['AY'])}: {row['ultimate']:,.0f}  [{lo:,.0f} ~ {hi:,.0f}]")

In [0]:
## 7. Visualizations
# Two charts that support the core message:
# **"Point estimates alone are not enough."**
# 7a. MSEP Decomposition — Where Does Uncertainty Come From?

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

ays = msep_df['AY'].astype(int).values
pv = msep_df['process_var'].values
parv = msep_df['parameter_var'].values

x = np.arange(len(ays))
width = 0.5

ax.bar(x, pv, width, label='Process Variance', color='#2196F3')
ax.bar(x, parv, width, bottom=pv, label='Parameter Variance', color='#FF9800')

ax.set_xlabel('Accident Year')
ax.set_ylabel('MSEP')
ax.set_title('MSEP Decomposition: Process vs Parameter Variance')
ax.set_xticks(x)
ax.set_xticklabels(ays)
ax.legend()
ax.ticklabel_format(style='plain', axis='y')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))

plt.tight_layout()
plt.show()

In [0]:
### 7b. Ultimate Loss ± 95% Confidence Interval
# The widening band for recent accident years shows that
# relying on a single point estimate understates the true risk.

fig, ax = plt.subplots(figsize=(10, 6))

ays = msep_df['AY'].astype(int).values
ultimates = msep_df['ultimate'].values

ci_lo = []
ci_hi = []
for _, row in msep_df.iterrows():
    lo, hi = compute_ci(row['ultimate'], row['msep'], level=0.95)
    ci_lo.append(lo)
    ci_hi.append(hi)

ci_lo = np.array(ci_lo)
ci_hi = np.array(ci_hi)

ax.plot(ays, ultimates, 'o-', color='#1565C0', linewidth=2, label='Ultimate (point estimate)')
ax.fill_between(ays, ci_lo, ci_hi, alpha=0.2, color='#1565C0', label='95% CI')

ax.set_xlabel('Accident Year')
ax.set_ylabel('Ultimate Loss')
ax.set_title('Ultimate Loss ± 95% Confidence Interval (Mack)')
ax.legend()
ax.ticklabel_format(style='plain', axis='y')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

In [0]:
## 8. Tail Factor Sensitivity (Optional)
# The analysis above uses tail factor = 1.0 (no tail), consistent with Medium-1.
# In practice, long-tail lines (bodily injury) may require tail > 1.0.
# Below shows the impact of a modest 1.03 tail factor on ultimate estimates.

tail = 1.03
print(f"Tail factor = {tail}\n")
print(f"{'AY':<6} {'Ultimate (no tail)':>20} {'Ultimate (with tail)':>22} {'Difference':>14}")
print("-" * 66)
for _, row in msep_df.iterrows():
    ult_tail = row["ultimate"] * tail
    diff = ult_tail - row["ultimate"]
    print(f"{int(row['AY']):<6} {row['ultimate']:>20,.0f} {ult_tail:>22,.0f} {diff:>14,.0f}")

In [0]:
## 9. Summary
# | What | Finding |
# |------|---------|
# | **Most uncertain AY** | Most recent (highest MSEP, widest CI) |
# | **Dominant variance source** | Process variance for immature AYs |
# | **Practical implication** | Reporting a single ultimate number without CI misrepresents the true risk position |
# This analysis extends Medium-1's chain-ladder pipeline by quantifying
# **how much we don't know** — not just what we estimate.